# 📚 Smart Library — Pipeline Streaming
## Notebook 04 : Événements temps réel → Apache Iceberg (merge-on-read)

---

### Objectif
Ingérer les **événements utilisateurs en temps réel**
et les stocker en Iceberg **merge-on-read** pour un traitement
quasi-instantané.

### Architecture streaming
```
                          Micro-batches Python (simulation)   Iceberg merge-on-read (MinIO)
──────────────            ──────────────────────────────      ─────────────────────────────
clics-utilisateurs   →    vagues successives              →   bronze.evenements_clics
recherches-util.     →    vagues successives              →   bronze.evenements_recherches
evenements-panier    →    vagues successives              →   bronze.evenements_panier
avis-livres          →    vagues successives              →   bronze.evenements_avis
                                  ↓
                          Agrégations Silver              →   silver.activite_temps_reel
                                  ↓
                          KPIs Gold temps réel             →   gold.tendances_temps_reel
```

### Concepts démontrés
- **Ingestion haute fréquence** avec Iceberg **merge-on-read**
- **Append incrémental** entre vagues d'événements (nouveau snapshot à chaque vague)
- **Fenêtres d'agrégation** pour les métriques temps réel
- **Fusion batch + streaming** : enrichissement avec les données Silver/Gold batch

> 💡 Le notebook Hudi original simule le streaming via des micro-batches Python plutôt
> que `readStream`/`writeStream` — cette version conserve la même approche pédagogique,
> avec des écritures Iceberg (`append`) à la place des upserts Hudi en mode `overwrite`.

---
## Étape 0 — Initialisation SparkSession

In [1]:
import os, sys, time, re
from datetime import datetime
from pathlib import Path

# ── Tuer toute session existante ─────────────────────────────
try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

# ── Java ──────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# ── Jars Iceberg + MinIO ──────────────────────────────────────
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.types import (
    StructType, StructField, IntegerType,
    DoubleType, BooleanType, LongType
)

# ── Connexion MinIO + Lakekeeper ──────────────────────────────
MINIO_ENDPOINT       = "http://minio:9000"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = "http://lakekeeper:8181/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

NS_BRONZE = f"{CATALOG_NAME}.bronze"
NS_SILVER = f"{CATALOG_NAME}.silver"
NS_GOLD   = f"{CATALOG_NAME}.gold"

# ── Fonctions utilitaires Iceberg (équivalent hudi_options) ───
#
#  Avec Hudi : le type de table (COW/MOR) et la clé d'upsert se
#  déclarent au niveau de l'ÉCRITURE (options du DataFrameWriter).
#  Avec Iceberg : ce sont des PROPRIÉTÉS DE TABLE, et l'UPSERT
#  s'exprime explicitement en SQL via MERGE INTO.
#
def iceberg_table_properties(type_table="copy-on-write"):
    """
    type_table :
      - "copy-on-write" → équivalent Hudi COW
      - "merge-on-read"  → équivalent Hudi MOR
    """
    return {
        "format-version"    : "2",
        "write.delete.mode" : type_table,
        "write.update.mode" : type_table,
        "write.merge.mode"  : type_table,
    }

def create_or_replace_table(df, table_name, partition_cols=None, type_table="copy-on-write"):
    """Équivalent de : df.write.format('hudi')....mode('overwrite').save(chemin)"""
    # Iceberg (ClusteredWriter) exige que les lignes soient triées/regroupées
    # par colonne de partition AVANT l'écriture — contrairement à Hudi qui
    # gère le clustering en interne. Sans ce tri : "Incoming records violate
    # the writer assumption that records are clustered by spec and partition".
    if partition_cols:
        df = df.sort(*partition_cols)
    writer = df.writeTo(table_name)
    if partition_cols:
        writer = writer.partitionedBy(*[F.col(c) for c in partition_cols])
    for k, v in iceberg_table_properties(type_table).items():
        writer = writer.tableProperty(k, v)
    writer.createOrReplace()

def append_table(df, table_name):
    """Ajout de lignes (insert simple, pas d'upsert)."""
    df.writeTo(table_name).append()

def upsert_table(spark, df, table_name, key_cols):
    """UPSERT via MERGE INTO — équivalent fonctionnel de l'UPSERT automatique Hudi."""
    vue = f"_staging_{table_name.split('.')[-1]}"
    df.createOrReplaceTempView(vue)
    condition = " AND ".join([f"t.{c} = s.{c}" for c in key_cols])
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {vue} s
        ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def snapshots(spark, table_name):
    """Retourne l'historique des snapshots Iceberg (équivalent de la timeline Hudi)."""
    return spark.sql(f"SELECT * FROM {table_name}.snapshots ORDER BY committed_at")

# ── SparkSession ──────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("SmartLibrary-Streaming")
    .master("local[4]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ── Création des namespaces Medallion (idempotent) ────────────
for ns in (NS_BRONZE, NS_SILVER, NS_GOLD):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ns}")

print(f"✅ SparkSession démarrée — version {spark.version}")
print(f"   Catalogue Iceberg : {CATALOG_NAME} (REST → Lakekeeper → Postgres)")
print(f"   Stockage          : {MINIO_ENDPOINT} (bucket lakehouse)")

# ── Tables streaming (Bronze) ─────────────────────────────────
BRONZE_CLICS       = f"{NS_BRONZE}.evenements_clics"
BRONZE_RECHERCHES  = f"{NS_BRONZE}.evenements_recherches"
BRONZE_PANIER      = f"{NS_BRONZE}.evenements_panier"
BRONZE_AVIS        = f"{NS_BRONZE}.evenements_avis"
SILVER_ACTIVITE    = f"{NS_SILVER}.activite_temps_reel"
GOLD_TENDANCES_RT  = f"{NS_GOLD}.tendances_temps_reel"

SILVER_UTILISATEURS = f"{NS_SILVER}.utilisateurs"
SILVER_LIVRES        = f"{NS_SILVER}.livres"
GOLD_TOP_LIVRES       = f"{NS_GOLD}.top_livres"

from pyspark.sql.window import Window

print(f"✅ SparkSession Streaming prête")

✅ SparkSession démarrée — version 3.4.0
   Catalogue Iceberg : lakehouse (REST → Lakekeeper → Postgres)
   Stockage          : http://minio:9000 (bucket lakehouse)
✅ SparkSession Streaming prête


---
## Étape 1 — Schémas des événements streaming

In [2]:
# ── Schéma : Clics utilisateurs ───────────────────────────────
schema_clics = StructType([
    StructField("event_id",    StringType(),  True),
    StructField("user_id",     StringType(),  True),
    StructField("book_id",     StringType(),  True),
    StructField("event_type",  StringType(),  True),
    StructField("source_page", StringType(),  True),
    StructField("duration_sec",IntegerType(), True),
    StructField("device",      StringType(),  True),
    StructField("timestamp",   StringType(),  True),
])

# ── Schéma : Recherches ───────────────────────────────────────
schema_recherches = StructType([
    StructField("event_id",        StringType(),  True),
    StructField("user_id",         StringType(),  True),
    StructField("query",           StringType(),  True),
    StructField("nb_resultats",    IntegerType(), True),
    StructField("filtre_genre",    StringType(),  True),
    StructField("filtre_langue",   StringType(),  True),
    StructField("filtre_prix_max", IntegerType(), True),
    StructField("clique_resultat", BooleanType(), True),
    StructField("timestamp",       StringType(),  True),
])

# ── Schéma : Panier ───────────────────────────────────────────
schema_panier = StructType([
    StructField("event_id",     StringType(),  True),
    StructField("user_id",      StringType(),  True),
    StructField("book_id",      StringType(),  True),
    StructField("action",       StringType(),  True),
    StructField("quantite",     IntegerType(), True),
    StructField("prix_unitaire",DoubleType(),  True),
    StructField("session_id",   StringType(),  True),
    StructField("timestamp",    StringType(),  True),
])

# ── Schéma : Avis ─────────────────────────────────────────────
schema_avis = StructType([
    StructField("review_id",     StringType(),  True),
    StructField("user_id",       StringType(),  True),
    StructField("book_id",       StringType(),  True),
    StructField("note",          IntegerType(), True),
    StructField("commentaire",   StringType(),  True),
    StructField("recommande",    BooleanType(), True),
    StructField("achat_verifie", BooleanType(), True),
    StructField("timestamp",     StringType(),  True),
])

print("✅ Schémas des données streaming définis")
print(f"   clics       : {len(schema_clics.fields)} champs")
print(f"   recherches  : {len(schema_recherches.fields)} champs")
print(f"   panier      : {len(schema_panier.fields)} champs")
print(f"   avis        : {len(schema_avis.fields)} champs")

✅ Schémas des données streaming définis
   clics       : 8 champs
   recherches  : 9 champs
   panier      : 8 champs
   avis        : 8 champs


---
## Étape 2 — Simulation des données streaming

> 💡 On génère des données directement en Python pour simuler des flux d'événements.

In [3]:
import random
from faker import Faker

fake = Faker("fr_FR")
random.seed(42)

# Charger les IDs depuis Silver
df_users_ref = (
    spark.table(SILVER_UTILISATEURS)
    .select("utilisateur_id")
    .limit(200)
)
df_livres_ref = (
    spark.table(SILVER_LIVRES)
    .select("livre_id", "genre", "prix_vente")
    .limit(200)
)

user_ids  = [r["utilisateur_id"] for r in df_users_ref.collect()]
livre_ids = [r["livre_id"]       for r in df_livres_ref.collect()]

print(f"✅ Référentiels chargés : {len(user_ids)} users, {len(livre_ids)} livres")

# ── Génération des événements ─────────────────────────────────
def generer_clics(n=500):
    return [{
        "event_id"    : f"CLK-{fake.uuid4()[:8]}",
        "user_id"     : random.choice(user_ids),
        "book_id"     : random.choice(livre_ids),
        "event_type"  : "click",
        "source_page" : random.choice(["accueil", "recherche", "categorie", "recommandation"]),
        "duration_sec": random.randint(3, 300),
        "device"      : random.choice(["mobile", "desktop", "tablette"]),
        "timestamp"   : datetime.now().isoformat(),
        "updated_at"  : datetime.now().isoformat(),
        "partition_device": random.choice(["mobile", "desktop", "tablette"]),
    } for _ in range(n)]

def generer_recherches(n=300):
    mots = ["roman", "histoire", "science", "amour", "guerre", "aventure", "polar"]
    return [{
        "event_id"       : f"SRC-{fake.uuid4()[:8]}",
        "user_id"        : random.choice(user_ids),
        "query"          : random.choice(mots) + " " + fake.word(),
        "nb_resultats"   : random.randint(0, 100),
        "filtre_genre"   : random.choice(["Roman", "SF", "Policier", None]),
        "filtre_langue"  : random.choice(["fr", "ar", None]),
        "filtre_prix_max": random.choice([20, 30, 50, None]),
        "clique_resultat": random.random() > 0.4,
        "timestamp"      : datetime.now().isoformat(),
        "updated_at"     : datetime.now().isoformat(),
        "partition_lang" : random.choice(["fr", "ar", "en"]),
    } for _ in range(n)]

def generer_panier(n=400):
    return [{
        "event_id"    : f"CRT-{fake.uuid4()[:8]}",
        "user_id"     : random.choice(user_ids),
        "book_id"     : random.choice(livre_ids),
        "action"      : random.choices(["ajout", "retrait", "achat"], weights=[60, 25, 15])[0],
        "quantite"    : random.randint(1, 3),
        "prix_unitaire": round(random.uniform(9.99, 49.99), 2),
        "session_id"  : f"SES-{random.randint(1000, 9999)}",
        "timestamp"   : datetime.now().isoformat(),
        "updated_at"  : datetime.now().isoformat(),
        "partition_action": random.choice(["ajout", "retrait", "achat"]),
    } for _ in range(n)]

def generer_avis(n=200):
    commentaires = [
        "Excellent livre, je recommande !",
        "Très bon, quelques longueurs.",
        "Moyen, pas mon style.",
        "Décevant par rapport aux attentes.",
        "Chef-d'œuvre absolu !"
    ]
    return [{
        "review_id"    : f"REV-{fake.uuid4()[:8]}",
        "user_id"      : random.choice(user_ids),
        "book_id"      : random.choice(livre_ids),
        "note"         : random.choices([1,2,3,4,5], weights=[5,8,15,35,37])[0],
        "commentaire"  : random.choice(commentaires),
        "recommande"   : random.random() > 0.3,
        "achat_verifie": random.random() > 0.4,
        "timestamp"    : datetime.now().isoformat(),
        "updated_at"   : datetime.now().isoformat(),
        "partition_note": str(random.choices([1,2,3,4,5], weights=[5,8,15,35,37])[0]),
    } for _ in range(n)]

print("✅ Générateurs d'événements prêts")

✅ Référentiels chargés : 200 users, 200 livres
✅ Générateurs d'événements prêts


---
## Étape 3 — Ingestion Streaming → Iceberg merge-on-read

In [4]:
# ── Ingestion des 4 flux d'événements ─────────────────────────
print("=" * 60)
print("  INGESTION STREAMING → ICEBERG MERGE-ON-READ")
print("=" * 60)

def ingerer_evenements(donnees, table_name, cle, partition):
    """
    Ingère un batch d'événements dans une table Iceberg merge-on-read.
    Première vague : création de la table. Vagues suivantes : append
    (chaque append crée un nouveau snapshot — pas de réécriture des
    données existantes, comme avec Hudi MOR).
    """
    df = spark.createDataFrame(donnees)
    df = df.withColumn("updated_at", F.col("updated_at").cast(StringType()))

    debut = time.time()
    if spark.catalog.tableExists(table_name):
        append_table(df, table_name)
    else:
        create_or_replace_table(df, table_name, partition_cols=[partition], type_table="merge-on-read")
    duree = round(time.time() - debut, 2)
    n = spark.table(table_name).count()
    print(f"  ✅ {table_name.split('.')[-1]:<35} {len(donnees):>4} événements → {duree}s")
    return n

# Batch 1 — ingestion initiale
print("\n  Batch 1 — Ingestion initiale")
print("-" * 60)
n_clics      = ingerer_evenements(generer_clics(500),      BRONZE_CLICS,      "event_id",  "partition_device")
n_recherches = ingerer_evenements(generer_recherches(300), BRONZE_RECHERCHES, "event_id",  "partition_lang")
n_panier     = ingerer_evenements(generer_panier(400),     BRONZE_PANIER,     "event_id",  "partition_action")
n_avis       = ingerer_evenements(generer_avis(200),       BRONZE_AVIS,       "review_id", "partition_note")

print(f"\n  Total événements ingérés : {n_clics + n_recherches + n_panier + n_avis:,}")

  INGESTION STREAMING → ICEBERG MERGE-ON-READ

  Batch 1 — Ingestion initiale
------------------------------------------------------------
  ✅ evenements_clics                     500 événements → 4.82s
  ✅ evenements_recherches                300 événements → 1.2s
  ✅ evenements_panier                    400 événements → 1.28s
  ✅ evenements_avis                      200 événements → 1.18s

  Total événements ingérés : 1,400


In [5]:
# ── Simulation : 3 micro-batches successifs ───────────────────
print("=" * 60)
print("  SIMULATION MICRO-BATCHES (3 vagues d'événements)")
print("=" * 60)

for batch_num in range(1, 4):
    print(f"\n  Batch {batch_num + 1} — {datetime.now().strftime('%H:%M:%S')}")
    print("-" * 60)

    # Nouveaux événements — chaque appel = un nouveau snapshot Iceberg
    ingerer_evenements(generer_clics(100),      BRONZE_CLICS,      "event_id",  "partition_device")
    ingerer_evenements(generer_recherches(80),  BRONZE_RECHERCHES, "event_id",  "partition_lang")
    ingerer_evenements(generer_panier(60),      BRONZE_PANIER,     "event_id",  "partition_action")
    ingerer_evenements(generer_avis(40),        BRONZE_AVIS,       "review_id", "partition_note")

    time.sleep(2)  # Pause entre les batches

# Bilan final
print("\n" + "=" * 60)
print("  ÉTAT FINAL DES TABLES MERGE-ON-READ")
print("=" * 60)
for nom, table in [
    ("bronze.evenements_clics",      BRONZE_CLICS),
    ("bronze.evenements_recherches", BRONZE_RECHERCHES),
    ("bronze.evenements_panier",     BRONZE_PANIER),
    ("bronze.evenements_avis",       BRONZE_AVIS),
]:
    n = spark.table(table).count()
    n_snaps = spark.sql(f"SELECT COUNT(*) AS n FROM {table}.snapshots").first()["n"]
    print(f"  {nom:<32} {n:>6,} événements  merge-on-read  ({n_snaps} snapshots) ✅")

  SIMULATION MICRO-BATCHES (3 vagues d'événements)

  Batch 2 — 23:19:35
------------------------------------------------------------
  ✅ evenements_clics                     100 événements → 0.83s
  ✅ evenements_recherches                 80 événements → 0.39s
  ✅ evenements_panier                     60 événements → 0.71s
  ✅ evenements_avis                       40 événements → 0.61s

  Batch 3 — 23:19:40
------------------------------------------------------------
  ✅ evenements_clics                     100 événements → 0.38s
  ✅ evenements_recherches                 80 événements → 0.42s
  ✅ evenements_panier                     60 événements → 0.48s
  ✅ evenements_avis                       40 événements → 0.38s

  Batch 4 — 23:19:44
------------------------------------------------------------
  ✅ evenements_clics                     100 événements → 0.35s
  ✅ evenements_recherches                 80 événements → 0.37s
  ✅ evenements_panier                     60 événements → 0.

---
## Étape 4 — Silver Temps Réel : Agrégations d'activité

> 💡 On agrège les événements ingérés pour produire des métriques
> d'activité en quasi-temps réel (équivalent des fenêtres de 5 minutes du pipeline Hudi).

In [6]:
# ── Lecture des événements Bronze ─────────────────────────────
df_clics = spark.table(BRONZE_CLICS)
df_rech  = spark.table(BRONZE_RECHERCHES)
df_pan   = spark.table(BRONZE_PANIER)
df_avis  = spark.table(BRONZE_AVIS)

# Enregistrer vues SQL
df_clics.createOrReplaceTempView("ev_clics")
df_rech.createOrReplaceTempView("ev_recherches")
df_pan.createOrReplaceTempView("ev_panier")
df_avis.createOrReplaceTempView("ev_avis")

print(f"✅ Événements chargés")
print(f"   Clics       : {df_clics.count():,}")
print(f"   Recherches  : {df_rech.count():,}")
print(f"   Panier      : {df_pan.count():,}")
print(f"   Avis        : {df_avis.count():,}")

✅ Événements chargés
   Clics       : 800
   Recherches  : 540
   Panier      : 580
   Avis        : 320


In [7]:
# ── Agrégation : livres les plus cliqués ─────────────────────
df_top_clics = spark.sql("""
    SELECT
        book_id                           AS livre_id,
        COUNT(*)                          AS nb_clics,
        COUNT(DISTINCT user_id)           AS nb_users_distincts,
        ROUND(AVG(duration_sec), 1)       AS duree_moyenne_sec,
        SUM(CASE WHEN device = 'mobile'
                 THEN 1 ELSE 0 END)       AS clics_mobile,
        SUM(CASE WHEN device = 'desktop'
                 THEN 1 ELSE 0 END)       AS clics_desktop,
        SUM(CASE WHEN source_page = 'recommandation'
                 THEN 1 ELSE 0 END)       AS clics_depuis_reco
    FROM ev_clics
    GROUP BY book_id
    ORDER BY nb_clics DESC
""")

print("📊 Top 10 livres les plus cliqués :")
df_top_clics.show(10, truncate=False)

# ── Agrégation : popularité des recherches ────────────────────
df_top_recherches = spark.sql("""
    SELECT
        query,
        COUNT(*)                           AS nb_recherches,
        ROUND(AVG(nb_resultats), 0)        AS resultats_moyens,
        ROUND(SUM(CASE WHEN clique_resultat
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS taux_clic_pct
    FROM ev_recherches
    GROUP BY query
    ORDER BY nb_recherches DESC
    LIMIT 10
""")

print("🔍 Top 10 recherches :")
df_top_recherches.show(10, truncate=False)

📊 Top 10 livres les plus cliqués :
+--------+--------+------------------+-----------------+------------+-------------+-----------------+
|livre_id|nb_clics|nb_users_distincts|duree_moyenne_sec|clics_mobile|clics_desktop|clics_depuis_reco|
+--------+--------+------------------+-----------------+------------+-------------+-----------------+
|BK0432  |10      |10                |145.0            |4           |4            |2                |
|BK0491  |10      |10                |131.7            |8           |1            |3                |
|BK0309  |9       |9                 |182.4            |7           |0            |4                |
|BK0357  |9       |9                 |165.2            |1           |5            |2                |
|BK0142  |9       |8                 |143.1            |4           |4            |0                |
|BK0454  |8       |8                 |172.9            |0           |3            |3                |
|BK0340  |8       |8                 |177.9    

In [8]:
# ── Activité par livre — fusion clics + panier + avis ─────────
df_activite = spark.sql("""
    SELECT
        COALESCE(c.livre_id, p.book_id, a.book_id)  AS livre_id,
        COALESCE(c.nb_clics, 0)                      AS nb_clics,
        COALESCE(p.nb_ajouts, 0)                     AS nb_ajouts_panier,
        COALESCE(p.nb_achats, 0)                     AS nb_achats_panier,
        COALESCE(a.nb_avis, 0)                       AS nb_avis_recents,
        COALESCE(a.note_moyenne, 0.0)                AS note_moyenne_recente,
        -- Score d'engagement temps réel
        ROUND(
            COALESCE(c.nb_clics, 0) * 1.0 +
            COALESCE(p.nb_ajouts, 0) * 3.0 +
            COALESCE(p.nb_achats, 0) * 5.0 +
            COALESCE(a.nb_avis, 0) * 2.0,
        2)                                           AS score_engagement_rt
    FROM (
        SELECT book_id AS livre_id, COUNT(*) AS nb_clics
        FROM ev_clics GROUP BY book_id
    ) c
    FULL OUTER JOIN (
        SELECT book_id,
               SUM(CASE WHEN action = 'ajout'  THEN 1 ELSE 0 END) AS nb_ajouts,
               SUM(CASE WHEN action = 'achat'  THEN 1 ELSE 0 END) AS nb_achats
        FROM ev_panier GROUP BY book_id
    ) p ON c.livre_id = p.book_id
    FULL OUTER JOIN (
        SELECT book_id,
               COUNT(*) AS nb_avis,
               ROUND(AVG(note), 2) AS note_moyenne
        FROM ev_avis GROUP BY book_id
    ) a ON COALESCE(c.livre_id, p.book_id) = a.book_id
    ORDER BY score_engagement_rt DESC
""")

# Ajouter métadonnées
df_activite_silver = (
    df_activite
    .withColumn("updated_at",     F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_livre",F.col("livre_id"))
)

print(f"✅ Activité temps réel calculée : {df_activite_silver.count()} livres")
print("\nTop 10 livres par engagement temps réel :")
df_activite_silver.select(
    "livre_id", "nb_clics", "nb_ajouts_panier",
    "nb_achats_panier", "nb_avis_recents",
    "note_moyenne_recente", "score_engagement_rt"
).show(10, truncate=False)

✅ Activité temps réel calculée : 200 livres

Top 10 livres par engagement temps réel :
+--------+--------+----------------+----------------+---------------+--------------------+-------------------+
|livre_id|nb_clics|nb_ajouts_panier|nb_achats_panier|nb_avis_recents|note_moyenne_recente|score_engagement_rt|
+--------+--------+----------------+----------------+---------------+--------------------+-------------------+
|BK0186  |5       |5               |2               |3              |4.33                |36.0               |
|BK0357  |9       |6               |0               |3              |3.33                |33.0               |
|BK0432  |10      |3               |1               |3              |5.0                 |30.0               |
|BK0183  |3       |3               |3               |1              |4.0                 |29.0               |
|BK0471  |2       |3               |2               |4              |4.0                 |29.0               |
|BK0187  |6       |3     

In [9]:
# ── Écriture Silver Activité Temps Réel (merge-on-read) ────────
create_or_replace_table(
    df_activite_silver, SILVER_ACTIVITE,
    partition_cols=["partition_livre"], type_table="merge-on-read"
)

n = spark.table(SILVER_ACTIVITE).count()
print(f"✅ {SILVER_ACTIVITE} — merge-on-read — {n} livres suivis")

✅ lakehouse.silver.activite_temps_reel — merge-on-read — 200 livres suivis


---
## Étape 5 — Gold Temps Réel : Fusion Batch + Streaming

> 💡 **Concept clé du Lakehouse** : on fusionne les données batch (ventes historiques
> de la couche Gold) avec les données streaming (engagement temps réel)
> pour produire un score global enrichi.

In [10]:
# ── Lecture Gold batch (top livres existants) ─────────────────
df_top_batch = (
    spark.table(GOLD_TOP_LIVRES)
    .select("livre_id", "titre", "genre", "auteur",
            "nb_exemplaires_vendus", "chiffre_affaires",
            "note_moyenne", "score_tendance", "performance")
)

# ── Lecture Silver activité temps réel ───────────────────────
df_activite_rt = (
    spark.table(SILVER_ACTIVITE)
    .select("livre_id", "nb_clics", "nb_ajouts_panier",
            "nb_achats_panier", "nb_avis_recents",
            "note_moyenne_recente", "score_engagement_rt")
)

# ── Fusion batch + streaming ──────────────────────────────────
df_gold_rt = (
    df_top_batch
    .join(df_activite_rt, on="livre_id", how="left")
    .fillna({
        "nb_clics": 0,
        "nb_ajouts_panier": 0,
        "nb_achats_panier": 0,
        "score_engagement_rt": 0.0
    })
    # Score unifié batch + streaming
    .withColumn("score_unifie",
                F.round(
                    F.col("score_tendance")     * 0.6 +
                    F.col("score_engagement_rt") * 0.4,
                2))
    # Rang unifié
    .withColumn("rang_unifie",
                F.rank().over(
                    Window.orderBy(F.col("score_unifie").desc())
                ))
    .withColumn("updated_at",      F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_genre", F.col("genre"))
)

print("🔥 TOP 10 — SCORE UNIFIÉ BATCH + STREAMING")
print("=" * 75)
(
    df_gold_rt
    .orderBy("rang_unifie")
    .select(
        "rang_unifie", "titre", "genre",
        "nb_exemplaires_vendus", "nb_clics",
        "nb_ajouts_panier", "score_tendance",
        "score_engagement_rt", "score_unifie"
    )
    .show(10, truncate=True)
)

🔥 TOP 10 — SCORE UNIFIÉ BATCH + STREAMING
+-----------+--------------------+--------------------+---------------------+--------+----------------+--------------+-------------------+------------+
|rang_unifie|               titre|               genre|nb_exemplaires_vendus|nb_clics|nb_ajouts_panier|score_tendance|score_engagement_rt|score_unifie|
+-----------+--------------------+--------------------+---------------------+--------+----------------+--------------+-------------------+------------+
|          1|             Horizon|Développement per...|                   30|       5|               5|          5.83|               36.0|        17.9|
|          2|Paupière Vaincre ...|            Histoire|                   46|      10|               3|           7.5|               30.0|        16.5|
|          3|               Matin|Développement per...|                   25|       9|               6|          2.67|               33.0|        14.8|
|          4|Air Poitrine Refu...|Développemen

In [11]:
# ── Écriture Gold Tendances Temps Réel ────────────────────────
create_or_replace_table(
    df_gold_rt, GOLD_TENDANCES_RT,
    partition_cols=["partition_genre"], type_table="copy-on-write"
)

n = spark.table(GOLD_TENDANCES_RT).count()
print(f"✅ {GOLD_TENDANCES_RT} — {n} livres avec score unifié")
print()
print("  💡 Ce score combine :")
print("     60% score_tendance     (ventes historiques batch)")
print("     40% score_engagement_rt (clics/panier/avis streaming)")

✅ lakehouse.gold.tendances_temps_reel — 500 livres avec score unifié

  💡 Ce score combine :
     60% score_tendance     (ventes historiques batch)
     40% score_engagement_rt (clics/panier/avis streaming)


---
## Étape 6 — Analyse des avis temps réel

In [12]:
# ── Analyse des notes en temps réel ──────────────────────────
print("⭐ ANALYSE DES AVIS TEMPS RÉEL")
print("=" * 50)

# Distribution des notes
spark.sql("""
    SELECT
        note,
        COUNT(*)                      AS nb_avis,
        ROUND(COUNT(*) * 100.0 /
              SUM(COUNT(*)) OVER(), 1) AS pourcentage,
        SUM(CASE WHEN recommande
                 THEN 1 ELSE 0 END)  AS nb_recommandations
    FROM ev_avis
    GROUP BY note
    ORDER BY note DESC
""").show()

# Livres les mieux notés (streaming)
print("Top 10 livres par note streaming :")
spark.sql("""
    SELECT
        book_id                         AS livre_id,
        COUNT(*)                        AS nb_avis,
        ROUND(AVG(note), 2)             AS note_moyenne,
        SUM(CASE WHEN recommande
                 THEN 1 ELSE 0 END)    AS nb_recommandations,
        SUM(CASE WHEN achat_verifie
                 THEN 1 ELSE 0 END)    AS avis_achats_verifies
    FROM ev_avis
    GROUP BY book_id
    HAVING COUNT(*) >= 2
    ORDER BY note_moyenne DESC, nb_avis DESC
    LIMIT 10
""").show(truncate=False)

⭐ ANALYSE DES AVIS TEMPS RÉEL
+----+-------+-----------+------------------+
|note|nb_avis|pourcentage|nb_recommandations|
+----+-------+-----------+------------------+
|   5|    118|       36.9|                94|
|   4|    120|       37.5|                79|
|   3|     41|       12.8|                25|
|   2|     20|        6.3|                15|
|   1|     21|        6.6|                15|
+----+-------+-----------+------------------+

Top 10 livres par note streaming :
+--------+-------+------------+------------------+--------------------+
|livre_id|nb_avis|note_moyenne|nb_recommandations|avis_achats_verifies|
+--------+-------+------------+------------------+--------------------+
|BK0432  |3      |5.0         |3                 |1                   |
|BK0198  |3      |5.0         |3                 |3                   |
|BK0045  |2      |5.0         |2                 |0                   |
|BK0004  |2      |5.0         |1                 |0                   |
|BK0043  |2     

---
## Étape 7 — Bilan du pipeline streaming

In [13]:
# ── Résumé complet ────────────────────────────────────────────
tables_streaming = [
    ("bronze.evenements_clics",      BRONZE_CLICS,      "MOR"),
    ("bronze.evenements_recherches", BRONZE_RECHERCHES, "MOR"),
    ("bronze.evenements_panier",     BRONZE_PANIER,     "MOR"),
    ("bronze.evenements_avis",       BRONZE_AVIS,       "MOR"),
    ("silver.activite_temps_reel",   SILVER_ACTIVITE,   "MOR"),
    ("gold.tendances_temps_reel",    GOLD_TENDANCES_RT, "COW"),
]

print("=" * 62)
print(f"  {'TABLE STREAMING':<34} {'MODE':<5} {'LIGNES':>8}")
print("=" * 62)
total = 0
for nom, table, type_t in tables_streaming:
    try:
        n = spark.table(table).count()
        total += n
        print(f"  {nom:<34} {type_t:<5} {n:>8,}  ✅")
    except Exception as e:
        print(f"  {nom:<34} {type_t:<5} {'ERREUR':>8}  ❌")
print("=" * 62)
print(f"  {'TOTAL':<34}       {total:>8,}")
print("=" * 62)
print()
print("  Concepts démontrés :")
print("  ✅ Iceberg merge-on-read — ingestion haute fréquence sans réécriture")
print("  ✅ Micro-batches   — 4 vagues d'événements simulées (4 snapshots/table)")
print("  ✅ Fusion batch+streaming — score unifié 60/40")
print("  ✅ Analyse temps réel    — engagement par livre")
print()
print("  ➡  Prochain : 05_time_travel_demo.ipynb")
print("     Time Travel avancé + Changelog + Rollback Iceberg")

  TABLE STREAMING                    MODE    LIGNES
  bronze.evenements_clics            MOR        800  ✅
  bronze.evenements_recherches       MOR        540  ✅
  bronze.evenements_panier           MOR        580  ✅
  bronze.evenements_avis             MOR        320  ✅
  silver.activite_temps_reel         MOR        200  ✅
  gold.tendances_temps_reel          COW        500  ✅
  TOTAL                                       2,940

  Concepts démontrés :
  ✅ Iceberg merge-on-read — ingestion haute fréquence sans réécriture
  ✅ Micro-batches   — 4 vagues d'événements simulées (4 snapshots/table)
  ✅ Fusion batch+streaming — score unifié 60/40
  ✅ Analyse temps réel    — engagement par livre

  ➡  Prochain : 05_time_travel_demo.ipynb
     Time Travel avancé + Changelog + Rollback Iceberg


In [14]:
spark.stop()
print("✅ SparkSession fermée — Pipeline Streaming complet !")

✅ SparkSession fermée — Pipeline Streaming complet !
